In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime
# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

#데이터 불러오기 
df = pd.read_csv('./model_df.csv')
print(df.columns)
df.head(1)

Index(['기획년도', '주차', '카테고리', '라인', '시즌이월', '시즌', '복종', '소품종', '성별', '총입고수량',
       '판매수량', '판매액', '평균택가', '평균원가', '총입고원가', '총입고택가', '매출원가계', '판매택가계',
       '주차별_평균_실판매가', '월', '월별_평균_실판매가', '시즌별_평균_실판매가', '실판매가', '할인율',
       '누적판매수량', '누적판매액', '누적매출원가', '누적판매택가', '누적판매율', 'ROI', '맑음', '흐림', '비',
       '강한 비', '눈', '강한 눈', '진눈깨비', '악천후일수', '평균기온(도)'],
      dtype='object')


,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,...,ROI,맑음,흐림,비,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도)
0,2021,2021-01-03,봄_가죽&FUR_가죽점퍼_ZA,ZA,01_시즌,봄,가죽&FUR,가죽점퍼,1:남성,1600,...,0.11,4,0,0,0,3,0,0,0,-9.1


In [ ]:
# 주차 기준 정렬 먼저
df = df.sort_values(['카테고리', '주차'])

# 할인율이 음수인 경우, 같은 카테고리 내에서 ffill
df['할인율'] = df.groupby('카테고리')['할인율'].transform(
    lambda x: x.mask(x < 0).ffill()
)
# df[df['할인율']<0]

,기획년도,주차,카테고리,라인,시즌이월,시즌,복종,소품종,성별,총입고수량,...,강한 비,눈,강한 눈,진눈깨비,악천후일수,평균기온(도),연도주차,가격_변화율,판매량_변화율,가격탄력성(판매량변화율/가격변화율)


In [3]:
# 피벗테이블로 4년동안 데이터가 있는 카테고리만 남김
# 카테고리별 연도 존재 여부 확인 (0: 없음, 1: 있음)
category_year_table = df.groupby(["카테고리", "기획년도"]).size().unstack(fill_value=0)
# 연도가 존재하면 1로 변환 (카테고리가 존재했음을 의미)
category_year_table = (category_year_table > 0).astype(int)

# 21, 22, 23, 24년도 모두 존재한 카테고리만 필터링
categories_all_years = category_year_table[
    (category_year_table.get(2021, 0) == 1) & 
    (category_year_table.get(2022, 0) == 1) & 
    (category_year_table.get(2023, 0) == 1) & 
    (category_year_table.get(2024, 0) == 1)
].index

# 새로운 데이터프레임 생성
df = df[df["카테고리"].isin(categories_all_years)].copy()
# df['카테고리'].nunique()

# 2023년도 데이터(지난 1년동안의 데이터)로 카테고리 선정하고자 함
# 판매 중간에 연도가 바뀌면서 잘린 겨울 제품 제외함함
df = df[(df['기획년도'] == 2023) & (df['시즌'] != '겨울')]
# df['카테고리'].nunique()

In [4]:
# 기준 1) ----------------------------------------------------------------------------------------------------------------
# 매출 기여도가 높은 핵심 카테고리 (판매량 & 매출 기준)
top_sales = df.groupby("카테고리", group_keys=False).agg(
    총판매수량=("판매수량", "sum"),
    총매출=("판매액", "sum"),
    총입고수량=("총입고수량", "max")
).reset_index()

# 기준 2) ----------------------------------------------------------------------------------------------------------------
# 재고 부담 & ROI 개선이 필요한 카테고리 (최대 누적판매율)
top_inventory_ROI = df.groupby("카테고리", group_keys=False).agg(
    최대누적판매율=("누적판매율", "max"),  
    평균할인율=("할인율", "mean"),
    누적판매액=("누적판매액", 'max'),
    누적매출원가=('누적매출원가', 'max'),
    총입고원가=('총입고원가', 'sum')
).reset_index()

top_inventory_ROI['ROI'] = ((top_inventory_ROI['누적판매액'] / 1.1 - top_inventory_ROI['누적매출원가']) / top_inventory_ROI['총입고원가']).round(2)

# 기준 3) ----------------------------------------------------------------------------------------------------------------
# 할인 전략 개선이 필요한 카테고리(평균 할인율)
# 시즌 종료 시 할인율 상승 패턴 확인
df["주차"] = pd.to_datetime(df["주차"], errors="coerce")  
df["연도주차"] = df["주차"].dt.strftime("%Y-%U")  

# 각 카테고리별 주차별 할인율 변화 측정
discount_trend = df.groupby(["카테고리", "연도주차"], group_keys=False)["할인율"].mean().reset_index()
discount_trend["할인율_변화"] = discount_trend.groupby("카테고리", group_keys=False)["할인율"].diff()  

# 마지막 4주 동안 할인율이 과하게 상승한 카테고리 찾기
last_weeks = discount_trend.groupby("카테고리", group_keys=False).tail(4)  
discount_rise = last_weeks.groupby("카테고리", group_keys=False)["할인율_변화"].sum().reset_index()
discount_rise.columns = ["카테고리", "시즌종료_할인율증가량"]

# 마지막 주차의 할인율 추가
last_week_discount = df.groupby("카테고리", group_keys=False).apply(
    lambda x: x.loc[x["주차"] == x["주차"].max(), "할인율"].mean()
).reset_index()
last_week_discount.columns = ["카테고리", "마지막주차_할인율"]

# 기준 4) ----------------------------------------------------------------------------------------------------------------
# 가격탄력성이 높은 카테고리 (할인율-판매량 상관관계)
top_correlation = df.groupby("카테고리", group_keys=False).apply(
    lambda x: x["할인율"].corr(x["판매수량"])
).reset_index()
top_correlation.columns = ["카테고리", "가격탄력성(할인율-판매량_상관관계)"]

# 가격_변화율과 판매량_변화율을 이용한 가격 탄력성 계산
df["가격_변화율"] = df.groupby('카테고리')['주차별_평균_실판매가'].pct_change() * 100
df["판매량_변화율"] = df.groupby('카테고리')['판매수량'].pct_change() * 100

df["가격탄력성(판매량변화율/가격변화율)"] = df["판매량_변화율"] / df["가격_변화율"]
df["가격탄력성(판매량변화율/가격변화율)"] = df["가격탄력성(판매량변화율/가격변화율)"].replace([np.inf, -np.inf], np.nan)

elastic_cate = df.groupby('카테고리')['가격탄력성(판매량변화율/가격변화율)'].mean().reset_index()

# 모든 데이터 병합 ----------------------------------------------------------------------------------------------------------------
category_selection = (
    top_sales
    .merge(top_inventory_ROI, on="카테고리", how="left")
    .merge(last_week_discount, on="카테고리", how="left")
    .merge(top_correlation, on="카테고리", how="left")
    .merge(discount_rise, on="카테고리", how="left")
    .merge(elastic_cate, on="카테고리", how="left")
    .fillna(0)  
)

# 소수점 라운딩
category_selection["평균할인율"] = category_selection["평균할인율"].round(2)
category_selection["가격탄력성(할인율-판매량_상관관계)"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].round(2)
category_selection["가격탄력성(판매량변화율/가격변화율)"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].round(2)
category_selection["마지막주차_할인율"] = category_selection["마지막주차_할인율"].astype(int)

# 정렬 및 순위 계산 (최대 누적판매율은 낮을수록 우선순위 → 오름차순, 나머지는 높을수록 우선순위 → 내림차순)
category_selection["총매출_순위"] = category_selection["총매출"].rank(method="min", ascending=False).astype(int)
category_selection["총입고_순위"] = category_selection["총입고수량"].rank(method="min", ascending=False).astype(int)
category_selection["최대누적판매율_순위"] = category_selection["최대누적판매율"].rank(method="min", ascending=True).astype(int)  # 낮을수록 재고 부담 ↑
category_selection["ROI_순위"] = category_selection["ROI"].rank(method="min", ascending=True).astype(int)  # 낮을수록 수익 손해 ↑
category_selection["평균할인율_순위"] = category_selection["평균할인율"].rank(method="min", ascending=False).astype(int)
category_selection["시즌종료_할인율증가량_순위"] = category_selection["시즌종료_할인율증가량"].rank(method="min", ascending=False).astype(int)
category_selection["가격탄력성(상관계수) 순위"] = category_selection["가격탄력성(할인율-판매량_상관관계)"].rank(method="min", ascending=False).astype(int) 
category_selection["가격탄력성(판매량변화율) 순위"] = category_selection["가격탄력성(판매량변화율/가격변화율)"].rank(method="min", ascending=True).astype(int) # 낮을수록 탄력성 좋음음

# 각 기준별 점수 계산 ----------------------------------------------------------------------------------------------------------------
# 기준 1: 매출 기여도 높은 카테고리 (총매출, 총입고수량)
category_selection["기준1_점수"] = category_selection["총매출_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["총입고_순위"].rank(method="min", ascending=True).astype(int)
# 기준 2: 재고 부담 & ROI 낮은 카테고리 (누적판매율, ROI)
category_selection["기준2_점수"] = category_selection["최대누적판매율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["ROI_순위"].rank(method="min", ascending=True).astype(int)

# 기준 3: 할인 전략 개선이 필요한 카테고리 (최대누적판매율, 평균할인율, 시즌종료_할인율증가량)
category_selection["기준3_점수"] = category_selection["평균할인율_순위"].rank(method="min", ascending=True).astype(int) + \
                                category_selection["시즌종료_할인율증가량_순위"].rank(method="min", ascending=True).astype(int)
     
# 기준 4: 가격탄력성이 높아 할인율 최적화 효율이 좋은 카테고리 (상관계수와 판매량변화율로 본 가격탄력성)
category_selection["기준4_점수"] = category_selection["가격탄력성(상관계수) 순위"].rank(method="min", ascending=True).astype(int) + \
                                 category_selection["가격탄력성(판매량변화율) 순위"].rank(method="min", ascending=True).astype(int)

# 기준별 순위 계산 (오름차순, 낮을수록 우선순위)
category_selection["기준1_순위"] = category_selection["기준1_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준2_순위"] = category_selection["기준2_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준3_순위"] = category_selection["기준3_점수"].rank(method="min", ascending=True).astype(int)
category_selection["기준4_순위"] = category_selection["기준4_점수"].rank(method="min", ascending=True).astype(int)

# 최종 순위 계산 (기준 1, 2, 3, 4 순위를 모두 합산)
category_selection["최종_순위"] = category_selection["기준1_순위"] + category_selection["기준2_순위"] + category_selection["기준3_순위"] + category_selection["기준4_순위"]
category_selection["최종_순위"] = category_selection["최종_순위"].rank(method="min", ascending=True).astype(int)

# 최종 출력: 순위만 포함 ----------------------------------------------------------------------------------------------------------------
final_rank_output = category_selection[[
    "카테고리","기준1_순위", "기준2_순위", "기준3_순위", "기준4_순위", "최종_순위","총판매수량","총매출","총입고수량","최대누적판매율","ROI","평균할인율","마지막주차_할인율",
    "시즌종료_할인율증가량","가격탄력성(할인율-판매량_상관관계)","가격탄력성(판매량변화율/가격변화율)","총매출_순위","총입고_순위","최대누적판매율_순위","ROI_순위",
    "평균할인율_순위","시즌종료_할인율증가량_순위","가격탄력성(상관계수) 순위","가격탄력성(판매량변화율) 순위"]].sort_values(by="최종_순위")

# final_rank_output #최종순위대로 정렬
# 기준 점수를 기준으로 정렬, 하나씩 주석 해제하면서 정렬
# final_rank_output.sort_values(by=["기준1_순위"])
# final_rank_output.sort_values(by=["기준2_순위"])
# final_rank_output.sort_values(by=["기준3_순위"])
final_rank_output.sort_values(by=["기준4_순위"])

,카테고리,기준1_순위,기준2_순위,기준3_순위,기준4_순위,최종_순위,총판매수량,총매출,총입고수량,최대누적판매율,...,가격탄력성(할인율-판매량_상관관계),가격탄력성(판매량변화율/가격변화율),총매출_순위,총입고_순위,최대누적판매율_순위,ROI_순위,평균할인율_순위,시즌종료_할인율증가량_순위,가격탄력성(상관계수) 순위,가격탄력성(판매량변화율) 순위
25,여름_우븐 셔츠_캐쥬얼셔츠_ZB,9,13,4,1,1,37990,1078654234,60865,62.42,...,0.49,-116.28,15,4,12,14,1,16,10,2
30,여름_팬츠_팬츠(일반)_ZB,2,24,11,1,2,70962,2586460323,102827,69.01,...,0.63,-21.62,6,2,21,20,10,13,6,6
12,사계절_수트_블레이져(수트)_ZB,2,11,25,3,6,30022,4490337431,43555,68.93,...,0.76,-8.92,1,7,20,2,18,28,1,13
26,여름_자켓_싱글재킷_ZA,6,19,9,4,2,20913,3129651456,30109,69.46,...,0.20,-1038.68,2,10,22,14,14,8,15,1
17,여름_수트_블레이져(수트)_ZA,18,12,25,5,13,6875,1443060061,10903,63.06,...,0.49,-14.49,14,24,14,9,21,25,10,7
13,사계절_수트_수트팬츠_ZB,5,4,23,6,2,32802,2594775689,52260,62.77,...,0.76,0.11,5,6,13,2,17,25,1,17
0,가을_니트 셔츠_라운드_ZB,29,23,18,7,24,3629,112643778,6310,57.51,...,0.52,-9.09,31,28,8,31,13,20,7,12
15,여름_니트 셔츠_라운드_ZB,4,27,3,8,7,78737,1999130498,100742,78.16,...,0.51,-9.73,7,3,29,20,5,8,9,11
2,가을_수트_수트팬츠_ZB,18,26,11,9,17,8637,843864022,12738,67.80,...,0.16,-55.13,18,20,19,25,22,1,17,4
10,봄_코트_싱글코트_ZB,28,1,5,10,9,2884,372321683,6009,47.99,...,0.11,-45.19,27,29,1,2,2,16,19,5


In [5]:
final_rank_output.to_csv("category_rank(소품종).csv", index=False, encoding="utf-8-sig")